# H-002 · 1D vs 1H after costs

**Universe C only. No 4H.** Do not fetch, cache, or simulate 4-hour bars.

Compare locked pairs under the H-001 trad-z OLS book: same per-fill `COSTS`, close-`t` signal → fill open `t+1`.
Scale OLS / z / HL lookbacks to **sessions** (`252/60/252` days × 6 hourly bars per session).

Sample: longest overlapping Yahoo 1H window (~730d). Walk-forward on the first 70% of that overlap; one sealed look on the last 30%.

After you type `BAR_STAR`:
- `"1d"` → later hyps use full C 1D with `RESEARCH_IS_END=2021-12-31`
- `"1h"` → later hyps stay on this window with 70/30 IS:OOS


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    fold_table,
    fold_val_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    ARTIFACTS_DIR,
    DEFAULT_STAR_STACK,
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_universe_c_panels,
    lookbacks_for_bar,
    overlay_kalman_hedge,
    repo_root,
    split_is_oos,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds
from strategies.s2_coint.config import S2SimConfig

STAR_PATH = DEFAULT_STAR_STACK
TEARSHEET_DIR = ARTIFACTS_DIR
stack = load_star_stack(STAR_PATH)
PAIRS_STAR = list(stack["PAIRS_STAR"])
print("stack keys:", sorted(stack))
print("PAIRS_STAR", PAIRS_STAR)


## 1. Load frozen 1D panel and fetch 1H (no 4H)


In [ ]:
from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.s2_coint_store import build_pair_panel
from backtest.s2_coint.research import (
    clip_panel_calendar,
    long_ohlcv_to_frames,
    overlap_calendar_bounds,
    overlap_is_end,
    pair_tuples,
    unique_tickers,
)

assert "4h" not in ("1d", "1h")
train_1d, full_1d = load_universe_c_panels("1d", PAIRS_STAR, root=ROOT)
tickers = unique_tickers(PAIRS_STAR)
print("tickers", tickers)

# Yahoo 1H history is short (~730d). Fetch from 2018 so the overlap is the 1H span.
frames_1h = {}
for t in tickers:
    long = fetch_ohlcv(t, "2018-01-01", interval="1h", isAsian=True)
    print(t, "1h rows", len(long), "min", long["date"].min() if len(long) else None, "max", long["date"].max() if len(long) else None)
    frames_1h.update(long_ohlcv_to_frames(long))

lb_1h = lookbacks_for_bar("1h")
panel_1h = build_pair_panel(
    frames_1h,
    pair_tuples(PAIRS_STAR),
    ols_window=lb_1h["ols_window"],
    z_window=lb_1h["z_window"],
    hl_window=lb_1h["hl_window"],
    include_adf_pvalue=True,
    include_variance_jump=True,
    hedge="ols",
)
start, end = overlap_calendar_bounds(full_1d["date"], panel_1h["date"])
panel_1d_ov = clip_panel_calendar(full_1d, start, end)
panel_1h_ov = clip_panel_calendar(panel_1h, start, end)
print("overlap", start, "→", end, "1d", len(panel_1d_ov), "1h", len(panel_1h_ov))

data_dir = os.path.join(ROOT, "01_data", "data_files", "s2_coint")
os.makedirs(data_dir, exist_ok=True)
panel_1h.to_parquet(os.path.join(data_dir, "s2_panel_C_1h_full.parquet"), index=False)


## 2. 70/30 split on the overlap window


In [ ]:
is_end_ov = overlap_is_end(panel_1d_ov, frac=0.70)
is_1d, oos_1d = split_is_oos(panel_1d_ov, is_end=is_end_ov)
is_1h, oos_1h = split_is_oos(panel_1h_ov, is_end=is_end_ov)
s1_weekly = load_s1_weekly(ROOT)
print("overlap IS end", is_end_ov, "1d IS", len(is_1d), "1h IS", len(is_1h))


## 3. Walk-forward folds (per bar)


In [ ]:
folds_1d = make_s2_folds(
    pd.DatetimeIndex(pd.to_datetime(is_1d["date"])).unique(),
    n_folds=3,
    embargo_bars=embargo_bars_for_config(bar="1d"),
)
folds_1h = make_s2_folds(
    pd.DatetimeIndex(pd.to_datetime(is_1h["date"])).unique(),
    n_folds=3,
    embargo_bars=embargo_bars_for_config(bar="1h"),
)
print("1D folds")
print(fold_table(folds_1d))
print("1H folds")
print(fold_table(folds_1h))


## 4. Fold-val metrics (validation only)


In [ ]:
cfg_1d = config_from_stack(stack, bar="1d")
cfg_1h = config_from_stack(stack, bar="1h")
df_1d = fold_val_metrics(is_1d, folds_1d, {"1d": cfg_1d}, s1_weekly=s1_weekly)
df_1h = fold_val_metrics(is_1h, folds_1h, {"1h": cfg_1h}, s1_weekly=s1_weekly)
fold_df = pd.concat([df_1d, df_1h], ignore_index=True)
fold_df


## 5. Boxplots (do not assign STAR here)


In [ ]:
plot_fold_boxplots(fold_df, title="H-002 1D vs 1H (no 4H)")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
fold_df.groupby("arm")[["ann_sharpe", "max_drawdown", "corr_to_s1"]].median()


## 6. Type `BAR_STAR` then save


In [ ]:
BAR_STAR = None  # TODO set after review: "1d" or "1h" — do not use argmax
require_star("BAR_STAR", BAR_STAR)
stack["BAR_STAR"] = BAR_STAR
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH)


## 7. Sealed OOS once (last 30% of overlap)


In [ ]:
require_star("BAR_STAR", BAR_STAR)
oos_panel = oos_1d if BAR_STAR == "1d" else oos_1h
oos_cfg = config_from_stack(stack, bar=BAR_STAR)
oos = run_s2_backtest(oos_panel, oos_cfg, s1_weekly=s1_weekly)
print(oos.metrics)
write_tearsheet_pdf(tearsheet_path("H-002", BAR_STAR), oos.returns, title=f"H-002 {BAR_STAR} sealed OOS")


## 8. Sample-rule fork for later hyps


If `BAR_STAR == "1d"`: later notebooks load `s2_panel_C_1d_{train,full}.parquet` and cut OOS at `2021-12-31`.

If `BAR_STAR == "1h"`: later notebooks load `s2_panel_C_1h_full.parquet` and use a 70/30 split on this overlap window. Optional: `s2_pair_panel_1h.ipynb` only if you need to rebuild the 1H panel.

H-003 is next (OLS vs Kalman). Never re-open `BAR_STAR`.


In [ ]:
if BAR_STAR == "1h":
    is_h, _ = split_is_oos(panel_1h_ov, is_end=is_end_ov)
    is_h.to_parquet(os.path.join(data_dir, "s2_panel_C_1h_train.parquet"), index=False)
    print("wrote 1h train parquet")
else:
    print("1D wins: later hyps use full C 1D research IS through 2021-12-31")
